In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.optimize import least_squares, root_scalar
from scipy import stats
import matplotlib.pyplot as plt

# ---- user input ----
USE_WEIGHTS = True
SIGMA_FLOOR = 0.01
T_FACTOR_95 = 1.96
N_STARTS = 20
LOO_N_STARTS = 3
P_PARTITION = 31.174
param_bounds = [(1, 100), (10, 10000), (10, 1000)]  # Vmax, Km, Ki

# ---- initial guesses (same as original second script's supplied helper) ----
def van_der_corput(n, base):
    vdc, denom = 0.0, 1.0
    while n:
        n, rem = divmod(n, base)
        denom *= base
        vdc += rem / denom
    return vdc


def generate_initial_guesses(n_points, bounds):
    h = np.zeros((n_points, len(bounds)))
    for i in range(n_points):
        h[i, 0] = i / n_points
        for j in range(1, len(bounds)):
            h[i, j] = van_der_corput(i, j + 1)
    return np.array([lo + (hi - lo) * h[:, j]
                     for j, (lo, hi) in enumerate(bounds)]).T


# ---- noncompetitive inhibition: original implicit model and solver ----
def implicit_integral(S, Vmax, Km, Ki, S0, t):
    A = Km * (1 + S0 / Ki)
    B = (1 + S0 / Ki) - Km / Ki
    C = -1.0 / Ki
    lhs = A * np.log(S) + B * S + 0.5 * C * S**2
    const0 = A * np.log(S0) + B * S0 + 0.5 * C * S0**2
    return lhs - (-Vmax * t + const0)


def solve_S(t, Vmax, Km, Ki, S0):
    guess = max(1e-12, S0 * np.exp(-Vmax * t / (Km + S0)))
    try:
        sol = root_scalar(implicit_integral, args=(Vmax, Km, Ki, S0, t),
                          x0=guess, x1=max(1e-12, 0.9 * S0), method="secant")
        if sol.converged:
            return sol.root
    except Exception:
        pass
    f_lo = implicit_integral(1e-12, Vmax, Km, Ki, S0, t)
    f_hi = implicit_integral(S0 * 0.999999, Vmax, Km, Ki, S0, t)
    if f_lo * f_hi < 0:
        sol = root_scalar(implicit_integral, args=(Vmax, Km, Ki, S0, t),
                          bracket=[1e-12, S0 * 0.999999], method="bisect")
        if sol.converged:
            return sol.root
    return np.nan


def model_fun(t, Vmax, Km, Ki, c0):
    S = np.array([solve_S(tt, Vmax, Km, Ki, c0) for tt in np.atleast_1d(t)])
    return c0 - S


def residuals(params, ts, Ps, c0s, sigmas):
    out = []
    for t, P, c0, sigma in zip(ts, Ps, c0s, sigmas):
        pred = model_fun(t, *params, c0)
        if not np.all(np.isfinite(pred)):
            return np.full(sum(len(x) for x in ts), 1e12)
        out.extend((pred - P) / sigma if USE_WEIGHTS else pred - P)
    return np.asarray(out)


lb, ub = map(np.array, zip(*param_bounds))


def fit_multistart(ts, Ps, c0s, sigmas, n_starts=N_STARTS):
    observed = np.concatenate(Ps)
    sigma = np.concatenate(sigmas)
    best, best_ssr = None, np.inf
    for p0 in generate_initial_guesses(n_starts, param_bounds):
        try:
            res = least_squares(residuals, p0, args=(ts, Ps, c0s, sigmas),
                                bounds=(lb, ub), method="trf", max_nfev=10000)
            if not res.success:
                continue
            pred = np.concatenate([model_fun(t, *res.x, c0) for t, c0 in zip(ts, c0s)])
            if not np.all(np.isfinite(pred)):
                continue
            ssr = np.sum((observed - pred)**2)  # original SSR-based selection
            if ssr < best_ssr:
                best, best_ssr = res, ssr
        except Exception:
            pass
    if best is None:
        raise RuntimeError("All multi-start fits failed")
    n, k = len(observed), len(best.x)
    chi2_red = np.sum(((observed - np.concatenate([
        model_fun(t, *best.x, c0) for t, c0 in zip(ts, c0s)])) / sigma)**2) / (n - k) if USE_WEIGHTS else np.nan
    aic = 2*k + n*np.log(best_ssr/n) if best_ssr > 0 else np.nan
    bic = k*np.log(n) + n*np.log(best_ssr/n) if best_ssr > 0 else np.nan
    return best, best_ssr, chi2_red, aic, bic


# ---- read relative data folder ----
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
data_folder = os.path.join(BASE_DIR, "data")
csv_files = sorted(glob.glob(os.path.join(data_folder, "*.csv")))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_folder}")

all_t, all_P, all_c0, all_sigma, order_info = [], [], [], [], []
for f in csv_files:
    df = pd.read_csv(f)
    if not {"t", "c_P", "c_S"}.issubset(df.columns):
        continue
    t = df["t"].to_numpy(float)
    cP = df["c_P"].to_numpy(float)
    c0 = float(df["c_S"].iloc[0])
    P = cP / P_PARTITION + cP  # original second script correction
    sigma = (np.maximum(SIGMA_FLOOR, df["u95_total"].to_numpy(float) / T_FACTOR_95)
             if USE_WEIGHTS and "u95_total" in df else np.ones_like(P))
    mask = np.isfinite(t) & np.isfinite(P) & np.isfinite(sigma)
    t, P, sigma = t[mask], P[mask], sigma[mask]
    if not len(t):
        continue
    all_t.append(t)
    all_P.append(P)
    all_c0.append(c0)
    all_sigma.append(sigma)
    order_info.append((os.path.basename(f).split("_")[0], f))

if not all_t:
    raise RuntimeError("No usable runs loaded")

n_runs = len(all_t)
P_concat = np.concatenate(all_P)
n_points = len(P_concat)

# ---- global fit ----
best_result, SSR, chi2_red, AIC, BIC = fit_multistart(all_t, all_P, all_c0, all_sigma)
P_pred = np.concatenate([model_fun(t, *best_result.x, c0) for t, c0 in zip(all_t, all_c0)])
k = len(best_result.x)
dof = n_points - k
SST = np.sum((P_concat - P_concat.mean())**2)
R2 = 1 - SSR/SST if SST > 0 else np.nan
R2_adj = 1 - (1 - R2)*(n_points - 1)/(n_points - k - 1) if n_points > k + 1 else np.nan
print(f"Loaded {n_runs} runs from {data_folder}")
print("==== Global fit (noncompetitive) ====")
print(f"Vmax={best_result.x[0]:.6g}, Km={best_result.x[1]:.6g}, Ki={best_result.x[2]:.6g}")
print(f"SSR={SSR:.6g}, MSE={SSR/n_points:.6g}, R2={R2:.6g}, R2_adj={R2_adj:.6g}")
print(f"Reduced chi2={chi2_red:.6g}, AIC={AIC:.6g}, BIC={BIC:.6g}")

# ---- leave-one-run-out validation ----
print("\n==== Leave-one-out over runs ====")
PRESS, n_val, chi2_val_total = 0.0, 0, 0.0
for i in range(n_runs):
    train = [j for j in range(n_runs) if j != i]
    if not train:
        break
    try:
        fit_i, _, _, aic_i, bic_i = fit_multistart(
            [all_t[j] for j in train], [all_P[j] for j in train],
            [all_c0[j] for j in train], [all_sigma[j] for j in train],
            n_starts=LOO_N_STARTS)
    except RuntimeError:
        continue
    pred_i = model_fun(all_t[i], *fit_i.x, all_c0[i])
    ssr_i = np.sum((all_P[i] - pred_i)**2)
    sst_i = np.sum((all_P[i] - all_P[i].mean())**2)
    r2_i = 1 - ssr_i/sst_i if sst_i > 0 else np.nan
    chi_i = np.sum(((all_P[i] - pred_i)/all_sigma[i])**2) if USE_WEIGHTS else np.nan
    PRESS += ssr_i
    n_val += len(all_P[i])
    if USE_WEIGHTS:
        chi2_val_total += chi_i
    print(f"[{i:02d}] {os.path.basename(order_info[i][1])}: R2_val={r2_i:.3f}, "
          f"chi2_red_val={chi_i/len(all_P[i]) if USE_WEIGHTS else np.nan:.3f}, "
          f"AIC_train={aic_i:.2f}, BIC_train={bic_i:.2f}")
if n_val:
    print(f"PRESS={PRESS:.6g}, RMSE_LOO={np.sqrt(PRESS/n_val):.6g}, "
          f"Q2={1-PRESS/SST if SST > 0 else np.nan:.6g}")
    if USE_WEIGHTS:
        print(f"chi2_red_LOO={chi2_val_total/n_val:.6g}")

# ---- Wald confidence intervals ----
try:
    cov = np.linalg.inv(best_result.jac.T @ best_result.jac) * (2*best_result.cost/dof)
    se = np.sqrt(np.diag(cov))
    t95 = stats.t.ppf(0.975, dof)
    print("\n95% Wald CIs:")
    for name, est, err in zip(("Vmax", "Km", "Ki"), best_result.x, se):
        print(f"{name}: {est:.5g} ± {t95*err:.5g} "
              f"({est-t95*err:.5g}, {est+t95*err:.5g})")
except np.linalg.LinAlgError:
    print("Wald CIs unavailable: singular covariance matrix")



# ---- plot ----
plt.figure(figsize=(5, 4))
for t, P in zip(all_t, all_P):
    plt.plot(t, P, "o", ms=3)
for t, c0 in zip(all_t, all_c0):
    tfit = np.linspace(float(np.min(t)), 5000, 200)
    plt.plot(tfit, model_fun(tfit, *best_result.x, c0), "-", lw=1)
plt.xlabel("time")
plt.ylabel("P")
plt.tight_layout()
plt.show()
